In [ ]:
import pandas as pd
from pathlib import Path

data_dir = Path.cwd() / "CAGDP2"

files = list(data_dir.glob("CAGDP2*ALL_AREAS*.csv"))

print("Matching files:")
for file in files:
    print(file.name)

In [ ]:
bea_file = files[0]

with open(bea_file, "r", encoding="utf-8-sig") as f:
    for _ in range(12):
        print(f.readline().strip())

In [ ]:
import pandas as pd

bea = pd.read_csv(
    bea_file,
    dtype={"GeoFIPS": str},
    encoding="cp1252",
    low_memory=False
)

print("Dataset shape:", bea.shape)

print("\nColumns:")
print(bea.columns.tolist())

print("\nFirst five rows:")
print(bea.head())

In [ ]:
bea["GeoFIPS"] = (
    bea["GeoFIPS"]
    .astype("string")
    .str.strip()
    .str.strip('"')
)

bea_total = bea[
    bea["LineCode"] == 1
].copy()

print("Total GDP records:", len(bea_total))

print("\nSample records:")
print(
    bea_total[
        ["GeoFIPS", "GeoName", "Description", "Unit", "2019", "2024"]
    ].head(15)
)

In [ ]:
print("GeoFIPS lengths:")
print(
    bea_total["GeoFIPS"]
    .str.len()
    .value_counts(dropna=False)
)

print("\nSample geographic records:")
print(
    bea_total[
        ["GeoFIPS", "GeoName"]
    ].head(25)
)

print("\nSample records ending in 000:")
print(
    bea_total[
        bea_total["GeoFIPS"].str.endswith("000", na=False)
    ][["GeoFIPS", "GeoName"]].head(15)
)

In [ ]:

bea_counties = bea_total[
    ~bea_total["GeoFIPS"].str.endswith("000", na=False)
].copy()

bea_counties = bea_counties[
    ~bea_counties["GeoFIPS"].str.startswith("72", na=False)
].copy()

print("County-level records:", len(bea_counties))

print("\nSample counties:")
print(bea_counties[["GeoFIPS", "GeoName"]].head(10))

print("\nDuplicate county FIPS:")
print(bea_counties["GeoFIPS"].duplicated().sum())

In [ ]:

years = ["2019", "2020", "2021", "2022", "2023", "2024"]

bea_long = bea_counties.melt(
    id_vars=["GeoFIPS", "GeoName"],
    value_vars=years,
    var_name="year",
    value_name="gdp_thousands"
)

bea_long = bea_long.rename(
    columns={
        "GeoFIPS": "county_fips",
        "GeoName": "county_name"
    }
)

bea_long["gdp_thousands"] = pd.to_numeric(
    bea_long["gdp_thousands"],
    errors="coerce"
)

bea_long["gdp_current_dollars"] = (
    bea_long["gdp_thousands"] * 1000
)

print("Total county-year records:", len(bea_long))

print("\nFirst five rows:")
print(bea_long.head())

print("\nMissing GDP values:")
print(bea_long["gdp_current_dollars"].isna().sum())

In [ ]:
print("Missing values:")
print(bea_long.isna().sum())

print("\nDuplicate county-year records:")
print(
    bea_long.duplicated(
        ["county_fips", "year"]
    ).sum()
)

print("\nRecords by year:")
print(
    bea_long["year"]
    .value_counts()
    .sort_index()
)

print("\nGDP summary statistics:")
print(
    bea_long["gdp_current_dollars"]
    .describe()
)

In [ ]:
missing_gdp = bea_long[
    bea_long["gdp_current_dollars"].isna()
].copy()

print("Missing GDP records:", len(missing_gdp))

print("\nMissing GDP values by year:")
print(
    missing_gdp["year"]
    .value_counts()
    .sort_index()
)

print("\nCounties with missing GDP values:")
print(
    missing_gdp[
        ["county_fips", "county_name", "year"]
    ].sort_values(["county_fips", "year"]).to_string(index=False)
)

In [ ]:
bea_counties["state_fips"] = (
    bea_counties["GeoFIPS"].str[:2]
)

print("County records by state FIPS:")
print(
    bea_counties["state_fips"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nTotal geographic records:")
print(len(bea_counties))

In [ ]:
missing_fips = missing_gdp["county_fips"].unique()

bea_missing_original = bea_total[
    bea_total["GeoFIPS"].isin(missing_fips)
].copy()

print(
    bea_missing_original[
        ["GeoFIPS", "GeoName", "2019", "2020",
         "2021", "2022", "2023", "2024"]
    ].to_string(index=False)
)

In [ ]:
aggregate_candidates = bea_total[
    bea_total["GeoFIPS"].str.endswith("000", na=False)
].copy()

print("Records ending in 000:", len(aggregate_candidates))

print(
    aggregate_candidates[
        ["GeoFIPS", "GeoName"]
    ].to_string(index=False)
)

In [ ]:
bea_clean = bea_long[
    [
        "county_fips",
        "county_name",
        "year",
        "gdp_thousands",
        "gdp_current_dollars"
    ]
].copy()

bea_clean["year"] = bea_clean["year"].astype(int)

bea_clean = bea_clean.sort_values(
    ["county_fips", "year"]
).reset_index(drop=True)

print("Final dataset shape:", bea_clean.shape)

print("\nFirst 10 records:")
print(bea_clean.head(10))

print("\nMissing GDP values:")
print(bea_clean["gdp_current_dollars"].isna().sum())

print("\nDuplicate county-year records:")
print(
    bea_clean.duplicated(
        ["county_fips", "year"]
    ).sum()
)

In [ ]:
from pathlib import Path

processed_dir = Path.cwd() / "processed"
processed_dir.mkdir(exist_ok=True)

output_file = processed_dir / "bea_county_gdp_2019_2024.csv"

bea_clean.to_csv(
    output_file,
    index=False
)

print("Saved cleaned dataset to:")
print(output_file)